In [1]:
import pandas as pd
import numpy as np


In [2]:
from google.colab import drive
import os
import sys

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Set directory path to your uploaded dataset location
data_dir = '/content/drive/MyDrive/AML_Dataset/AML_Dataset'

# 3. Import temporal split logic from split_data.py
if '/content' not in sys.path:
    sys.path.append('/content')

from split_data import temporal_split

Mounted at /content/drive


In [3]:
# Load the dataset using the Google Drive path
print("Loading dataset...")
csv_path = os.path.join(data_dir, 'HI-Small_Trans.csv')
df = pd.read_csv(csv_path)
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

Loading dataset...


In [4]:
# 1. Apply temporal split FIRST to prevent temporal leakage
train_df, test_df = temporal_split(df)

# Function to run feature engineering cleanly on split data
def engineer_features(data):
    df_out = data.copy()

    # Existing basic flags
    df_out['is_cross_border'] = (df_out['From Bank'] != df_out['To Bank']).astype(int)
    df_out['is_cross_currency'] = (df_out['Receiving Currency'] != df_out['Payment Currency']).astype(int)
    df_out['is_round_amount'] = (df_out['Amount Paid'] % 1000 == 0).astype(int)

    # Missing Domain Risk Features (ACH & SAR)
    df_out['is_ach'] = (df_out['Payment Format'] == 'ACH').astype(int)
    df_out['is_ach_sar'] = (
        (df_out['Payment Format'] == 'ACH') &
        ((df_out['Receiving Currency'] == 'Saudi Riyal') | (df_out['Payment Currency'] == 'Saudi Riyal'))
    ).astype(int)

    # Account Aggregations
    tx_counts = df_out.groupby('Account').size().reset_index(name='tx_count_out')
    df_out = df_out.merge(tx_counts, on='Account', how='left')

    unique_counterparties = df_out.groupby('Account')['Account.1'].nunique().reset_index(name='unique_counterparties_out')
    df_out = df_out.merge(unique_counterparties, on='Account', how='left')

    # Burstiness (24h Window)
    df_out = df_out.sort_values('Timestamp')
    df_indexed = df_out.set_index('Timestamp')
    rolling_counts = df_indexed.groupby('Account').rolling('24h')['Amount Paid'].count().reset_index(name='tx_24h_burst')
    df_out = df_out.merge(rolling_counts, on=['Account', 'Timestamp'], how='left')

    # In/Out Ratios
    account_totals = df_out.groupby('Account').agg(
        total_paid=('Amount Paid', 'sum'),
        total_received=('Amount Received', 'sum')
    ).reset_index()
    account_totals['in_out_ratio'] = account_totals['total_received'] / (account_totals['total_paid'] + 1)
    df_out = df_out.merge(account_totals[['Account', 'in_out_ratio']], on='Account', how='left')

    df_out.fillna(0, inplace=True)
    return df_out

# Execute feature pipeline
train_engineered = engineer_features(train_df)
test_engineered = engineer_features(test_df)

# Export leakage-free datasets for Stage 3 baseline modeling
train_engineered.to_parquet(os.path.join(data_dir, 'train_features.parquet'), index=False)
test_engineered.to_parquet(os.path.join(data_dir, 'test_features.parquet'), index=False)
print("Leakage-free feature tables successfully saved to Drive.")

Leakage-free feature tables successfully saved to Drive.


In [5]:

# 1. Cross-Border & Cross-Currency Flags
print("Engineering cross-border and currency flags...")
df['is_cross_border'] = (df['From Bank'] != df['To Bank']).astype(int)
df['is_cross_currency'] = (df['Receiving Currency'] != df['Payment Currency']).astype(int)


Engineering cross-border and currency flags...


In [6]:

# 2. Account-level Aggregations
print("Calculating account-level aggregations...")
tx_counts = df.groupby('Account').size().reset_index(name='tx_count_out')
df = df.merge(tx_counts, on='Account', how='left')

unique_counterparties = df.groupby('Account')['Account.1'].nunique().reset_index(name='unique_counterparties_out')
df = df.merge(unique_counterparties, on='Account', how='left')


Calculating account-level aggregations...


In [7]:

# 3. Rolling Time Windows (Burstiness)
print("Computing rolling window burstiness...")
df = df.sort_values('Timestamp')
df_indexed = df.set_index('Timestamp')
rolling_counts = df_indexed.groupby('Account').rolling('24h')['Amount Paid'].count().reset_index(name='tx_24h_burst')
df = df.merge(rolling_counts, on=['Account', 'Timestamp'], how='left')


Computing rolling window burstiness...


In [8]:

# 4. Round-Number Flags
print("Flagging round amounts...")
df['is_round_amount'] = (df['Amount Paid'] % 1000 == 0).astype(int)


Flagging round amounts...


In [9]:

# 5. In/Out Ratio
print("Calculating in/out ratios...")
account_totals = df.groupby('Account').agg(
    total_paid=('Amount Paid', 'sum'),
    total_received=('Amount Received', 'sum')
).reset_index()
account_totals['in_out_ratio'] = account_totals['total_received'] / (account_totals['total_paid'] + 1)
df = df.merge(account_totals[['Account', 'in_out_ratio']], on='Account', how='left')


Calculating in/out ratios...


In [10]:

# 6. Fill missing values that might result from rolling windows
df.fillna(0, inplace=True)


In [11]:
# 7. Export the Final Feature Table
print("Exporting feature table...")
output_parquet_path = os.path.join(data_dir, 'features_stage2.parquet')
df.to_parquet(output_parquet_path, index=False)
print(f"Feature Engineering Complete. Table saved to {output_parquet_path}")

Exporting feature table...
Feature Engineering Complete. Table saved to /content/drive/MyDrive/AML_Dataset/AML_Dataset/features_stage2.parquet
